In [ ]:
!pip install playwright pandas
!playwright install chromium
!pip install nest_asyncio

In [ ]:
!apt-get update

!apt-get install -y \
    libatk1.0-0 \
    libatk-bridge2.0-0 \
    libcups2 \
    libxkbcommon0 \
    libxcomposite1 \
    libxdamage1 \
    libxrandr2 \
    libgbm1 \
    libasound2 \
    libpango-1.0-0 \
    libcairo2 \
    libatspi2.0-0 \
    libgtk-3-0

In [ ]:
import asyncio
import nest_asyncio
import re
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse, parse_qs
from playwright.async_api import async_playwright

nest_asyncio.apply()

VIDEO_URL = "https://www.tiktok.com/@niickjackson/video/7619530585454808334"
OUTPUT_FILE = "tiktok_comments_@niickjackson_video_7619530585454808334_10.csv"

MAX_COMMENTS = 300
HEADLESS = True
SCROLL_PAUSE = 0.5
MAX_SCROLLS = 50

COLUMNS = [
    "platform", "content_id", "video_id", "comment_id",
    "comment_text", "likes", "created_at", "scraped_at"
]


def extract_video_id(url: str) -> str:
    match = re.search(r"/video/(\d+)", url)
    if not match:
        raise ValueError("Nu am gasit video_id in URL.")
    return match.group(1)


def parse_comment(comment: dict, video_id: str) -> dict | None:
    text = comment.get("text", "").strip()
    if not text:
        return None

    return {
        "platform": "tiktok",
        "content_id": f"tiktok_{video_id}",
        "video_id": video_id,
        "comment_id": comment.get("cid"),
        "comment_text": text,
        "likes": comment.get("digg_count", 0),
        "created_at": comment.get("create_time"),
        "scraped_at": datetime.now().isoformat(),
    }


async def scrape_comments(video_url: str, max_comments: int = 100) -> list[dict]:
    video_id = extract_video_id(video_url)
    collected = {}
    done = asyncio.Event()

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=HEADLESS,
            args=[
                "--no-sandbox",
                "--disable-dev-shm-usage",
                "--disable-blink-features=AutomationControlled",
            ],
        )

        context = await browser.new_context(
            viewport={"width": 1280, "height": 800},
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            ),
            locale="ro-RO",
        )

        page = await context.new_page()

        async def on_response(response):
            if "comment/list" not in response.url:
                return

            query = parse_qs(urlparse(response.url).query)
            aweme_id = query.get("aweme_id", [""])[0]
            request_cursor = query.get("cursor", [""])[0]

            if aweme_id != video_id:
                print(f"Ignore request for another video: {aweme_id}")
                return

            print("\nRequest comments for the correct video:")
            print("cursor request:", request_cursor)

            try:
                body = await response.json()
            except Exception as e:
                print("I cannot parse JSON:", e)
                return

            comments_batch = body.get("comments") or []

            print("status_code:", body.get("status_code"))
            print("has_more:", body.get("has_more"))
            print("cursor response:", body.get("cursor"))
            print("batch size:", len(comments_batch))

            for c in comments_batch:
                row = parse_comment(c, video_id)
                if row:
                    key = row["comment_id"] or row["comment_text"]
                    collected[key] = row

            print(f"Unique comments collected: {len(collected)}")

            if len(collected) >= max_comments:
                done.set()

        page.on("response", on_response)

        print(f"Deschidem: {video_url}")

        await page.goto(video_url, wait_until="domcontentloaded", timeout=60_000)
        await asyncio.sleep(8)

        try:
            comment_button = page.locator(
                "[data-e2e='comment-icon'], "
                "[aria-label*='comment'], "
                "[aria-label*='Comment'], "
                "button:has-text('Comment'), "
                "button:has-text('Comentariu'), "
                "button:has-text('Comentarii')"
            ).first

            await comment_button.click(timeout=10_000)
            print("I have clicked the comments button.")
            await asyncio.sleep(5)

        except Exception as e:
            print("I could not automatically click the comments button:", e)

        await page.screenshot(path="after_comment_click.png", full_page=True)
        print("Screenshot saved: after_comment_click.png")

        scroll_count = 0
        last_count = len(collected)
        no_change_count = 0

        while not done.is_set() and scroll_count < MAX_SCROLLS:

            await page.mouse.move(1010, 500)
            await page.mouse.click(1010, 500)
            await page.mouse.wheel(0, 1200)

            await asyncio.sleep(SCROLL_PAUSE)

            scroll_count += 1
            current_count = len(collected)

            print(f"Scroll #{scroll_count} — collected comments: {current_count}")

            if current_count == last_count:
                no_change_count += 1
            else:
                no_change_count = 0

            last_count = current_count

            if no_change_count >= 20:
                print("No more new comments after 20 scrolls.")
                break

        await browser.close()

    return list(collected.values())[:max_comments]


def main():
    comments = asyncio.run(scrape_comments(VIDEO_URL, MAX_COMMENTS))

    df = pd.DataFrame(comments, columns=COLUMNS)

    if not df.empty and "created_at" in df.columns:
        df["created_at"] = pd.to_datetime(
            df["created_at"],
            unit="s",
            errors="coerce",
        ).dt.strftime("%Y-%m-%d %H:%M:%S")

    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\nSaved {len(comments)} comments in {OUTPUT_FILE}")

    if not df.empty:
        print("\nFirst comments collected:")
        print(df[["comment_text", "likes", "created_at"]].head(10).to_string(index=False))


if __name__ == "__main__":
    main()

Lucrare la Azure

In [ ]:
# Install Microsoft ODBC 17 for SQL Server
!curl https://packages.microsoft.com/keys/microsoft.asc | sudo apt-key add -
!curl https://packages.microsoft.com/config/ubuntu/$(lsb_release -rs)/prod.list > /etc/apt/sources.list.d/mssql-release.list
!sudo apt-get update
!sudo ACCEPT_EULA=Y apt-get install -y msodbcsql17

In [ ]:
!pip install pyodbc sqlalchemy

In [ ]:
!pip install azure-cosmos

In [ ]:
from azure.cosmos import CosmosClient
import pandas as pd

URL = "https://proiectnlp.documents.azure.com:443/"
KEY = "=========KEY========="
DATABASE_NAME = "base_nlp"
#CONTAINER_NAME = "C3_TK" #
CONTAINER_NAME = "Video"

try:
    # 1.Client
    client = CosmosClient(URL, credential=KEY)

    # 2. Access DB & Container
    database = client.get_database_client(DATABASE_NAME)
    container = database.get_container_client(CONTAINER_NAME)

    query = "SELECT * FROM c"
    items = list(container.query_items(query=query, enable_cross_partition_query=True))

    df = pd.DataFrame(items)

    # Clean columns that Cosmos adds automatically (optional)
    #cols_to_drop = ['_rid', '_self', '_etag', '_attachments', '_ts']
    #df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    print("Succes!")
    display(df)

except Exception as e:
    print(f"Error connecting to Cosmos DB: {e}")

Succes! Datele au fost extrase din Cosmos DB.


,id,video_url,category,_rid,_self,_etag,_attachments,_ts
0,968be112-eb9a-4eaa-81dc-e26fc34f1b3e,https://www.youtube.com/shorts/968be112-eb9a-4...,gaming,VyV0AK6R9JUzAAAAAAAAAA==,dbs/VyV0AA==/colls/VyV0AK6R9JU=/docs/VyV0AK6R9...,"""5200a62a-0000-0e00-0000-69f114000000""",attachments/,1777406976
1,57c22ce0-6382-4d27-b37f-c068eab7c71a,https://www.youtube.com/shorts/57c22ce0-6382-4...,gaming,VyV0AK6R9JU0AAAAAAAAAA==,dbs/VyV0AA==/colls/VyV0AK6R9JU=/docs/VyV0AK6R9...,"""5200bf2a-0000-0e00-0000-69f114340000""",attachments/,1777407028
2,8557c8f2-7fb1-419e-b3be-f9187c1ed2d2,https://www.youtube.com/shorts/8557c8f2-7fb1-4...,gaming,VyV0AK6R9JU1AAAAAAAAAA==,dbs/VyV0AA==/colls/VyV0AK6R9JU=/docs/VyV0AK6R9...,"""5200da2a-0000-0e00-0000-69f114670000""",attachments/,1777407079
3,485d1c8c-b42e-499b-8c71-8ec7cdafcaa5,https://www.youtube.com/shorts/485d1c8c-b42e-4...,gaming,VyV0AK6R9JU2AAAAAAAAAA==,dbs/VyV0AA==/colls/VyV0AK6R9JU=/docs/VyV0AK6R9...,"""5200e12a-0000-0e00-0000-69f1149d0000""",attachments/,1777407133
4,6de42e2a-a943-475d-afa5-4b0880fd443d,https://www.youtube.com/shorts/6de42e2a-a943-4...,gaming,VyV0AK6R9JU3AAAAAAAAAA==,dbs/VyV0AA==/colls/VyV0AK6R9JU=/docs/VyV0AK6R9...,"""5200f22a-0000-0e00-0000-69f114d40000""",attachments/,1777407188
...,...,...,...,...,...,...,...,...
134,8d806655-ecb4-4491-a25a-331a070c2f39,https://www.instagram.com/reel/DEbxwMJx-Og/,comedy,VyV0AK6R9JW5AAAAAAAAAA==,dbs/VyV0AA==/colls/VyV0AK6R9JU=/docs/VyV0AK6R9...,"""450027b3-0000-0e00-0000-69f50b620000""",attachments/,1777666914
135,cef81a84-9eb0-42f5-8c9d-cce87143b882,https://www.instagram.com/reel/C_6w5ybRAuG/,comedy,VyV0AK6R9JW6AAAAAAAAAA==,dbs/VyV0AA==/colls/VyV0AK6R9JU=/docs/VyV0AK6R9...,"""450028b3-0000-0e00-0000-69f50b630000""",attachments/,1777666915
136,62299701-aa24-401a-a09a-e7d8f3526c89,https://www.instagram.com/reel/DEm_j7WRsHV/,comedy,VyV0AK6R9JW7AAAAAAAAAA==,dbs/VyV0AA==/colls/VyV0AK6R9JU=/docs/VyV0AK6R9...,"""450029b3-0000-0e00-0000-69f50b630000""",attachments/,1777666915
137,438622bc-70dc-4215-b5a2-f293ff090c06,https://www.instagram.com/reel/DEoDDltxbYD/,comedy,VyV0AK6R9JW8AAAAAAAAAA==,dbs/VyV0AA==/colls/VyV0AK6R9JU=/docs/VyV0AK6R9...,"""45002ab3-0000-0e00-0000-69f50b630000""",attachments/,1777666915


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Procesarea datelor

In [ ]:
import pandas as pd
import os

base_path = "drive/MyDrive/Fisiere_Proiect_NLP"

comedy_directory = f'{base_path}/Comedy'
educatie_directory = f'{base_path}/Educatie'
gaming_directory = f'{base_path}/Gaming'
sport_directory = f'{base_path}/Sport'
tehnologie_directory = f'{base_path}/Tehnologie'

for directory in [comedy_directory, educatie_directory, gaming_directory, sport_directory, tehnologie_directory]:
    for filename in os.listdir(directory):
        if filename.endswith('csv'):
          print(f'Processing file: {filename}')
          df = pd.read_csv(os.path.join(directory, filename))
          df.head()

Processing file: tiktok_comments_@jfddxgzhf_video_7585936705199426847_5_filtrat.csv
Processing file: tiktok_comments_@uzkofbqd0aj_video_7624875235690548510_1_filtrat.csv
Processing file: tiktok_comments_@dailylols7_video_7623070341316775199_4_filtrat.csv
Processing file: tiktok_comments_@nne_hub_video_7626311029684784397_3_filtrat.csv
Processing file: tiktok_comments_@johnnyviti_video_7625753695480761613_7_filtrat.csv
Processing file: tiktok_comments_@jordhaley_video_7622800699683867935_9_filtrat.csv
Processing file: tiktok_comments_@wrldst4rr.haydenn_video_7621991999251467550_2_filtrat.csv
Processing file: tiktok_comments_@k.prezzi_video_7623182373315775757_6_filtrat.csv
Processing file: tiktok_comments_@astroathens_video_6808960524206869766_3_filtrat.csv
Processing file: tiktok_comments_@idea.soup_video_6941178518323612933_2_filtrat.csv
Processing file: tiktok_comments_@mathswithmisschang_video_6858581045965704453_6_filtrat.csv
Processing file: tiktok_comments_@lawbymike_video_762418

In [ ]:
import os
import pandas as pd
import uuid
import urllib
from azure.cosmos import CosmosClient
from google.colab import drive

drive.mount('/content/drive')

URL = "https://proiectnlp.documents.azure.com:443/"
KEY = "=========KEY========="

client = CosmosClient(URL, credential=KEY)
database = client.get_database_client("base_nlp")

video_container = database.get_container_client("Video")
comment_container = database.get_container_client("C3_TK")

base_path = "/content/drive/MyDrive/Fisiere_Proiect_NLP"
directories = {
    #'Comedy': f'{base_path}/Comedy',
    #'education': f'{base_path}/education',
    #'gaming': f'{base_path}/gaming',
    #'sport': f'{base_path}/sport',
    #'tech': f'{base_path}/tech'
}

# 4. Process + Insertion
for category, directory in directories.items():
    if not os.path.exists(directory):
        print(f"Sărit peste: {category} (folderul nu există)")
        continue

    for filename in os.listdir(directory):
        if filename.endswith('.csv'):
            print(f'--- Procesare fișier: {filename} ({category}) ---')
            file_path = os.path.join(directory, filename)

            try:
                df = pd.read_csv(file_path)

                match = re.search(r'(@\w+)_video_(\d+)', file_path)

                if match:
                    username = match.group(1)
                    video_id = match.group(2)
                    url = f"https://www.tiktok.com/{username}/video/{video_id}"
                    #print(url)

                    #Insert VIDEO
                    video_id = str(uuid.uuid4())

                    video_data = {
                        "id": video_id,
                        "video_url": url,
                        "category": category
                    }
                    video_container.upsert_item(video_data)
                    print(f"Video inserat cu succes. ID: {video_id}")

                    #Insert COMMENTS
                    for index, row in df.iterrows():
                        comment_text = row['comment_text'] if 'comment_text' in df.columns else "Text lipsa"

                        comment_data = {
                            "id": str(uuid.uuid4()),
                            "text": comment_text,
                            "label": 3,
                            "video_id": video_id,
                        }

                        comment_container.upsert_item(comment_data)

                    print(f"Inserted {len(df)} comments.")

            except Exception as e:
                print(f"Error {filename}: {e}")

print("\nFinished!")